# Module 13: Spillover and Contamination

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every other failure in this series makes a program look **better** than it
was. This one makes it look worse, which is why a program that genuinely
works can be cancelled after a careful evaluation.

It is also the only bias that cannot be found in the data. You have to ask.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
prof = profile.set_index("agency_id")

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated_ids, outcome="n_uof", offset=None):
    """The standard specification, with whoever is labelled treated."""
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated_ids))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated_ids))
                  & (s["period"] == "phase")).astype(float)
    off = s["lo"] if offset is None else offset
    z = smf.glm(f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase", s,
                family=sm.families.Poisson(), offset=off).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z.bse["settled"]

## 2. One comparison agency, secretly trained

Suppose one agency in the comparison group had in fact run the same training,
under another name, and nobody recorded it.

In [ ]:
clean = fit(d, keep)[0]
rows = []
for a in sorted(COMPARISON, key=lambda x: -prof.loc[x, "sworn_officers"]):
    e = fit(d, keep + [a])[0]
    rows.append({"agency also trained": NAME[a],
                 "sworn officers": int(prof.loc[a, "sworn_officers"]),
                 "estimate becomes": f"{e:+.1f}%",
                 "points lost": round(abs(e - clean), 1)})
print(f"  correctly labelled: {clean:+.1f}%   the truth: {TRUTH:+.1f}%\n")
pd.DataFrame(rows).set_index("agency also trained")

**The damage tracks the agency's size, not its similarity.** Ashfell, with
902 officers, costs four percentage points. Orrindale, with eight, costs
nothing measurable.

The reason is the same one from [Module 4](Module_04_Building_A_Comparison_Group.ipynb):
the pooled comparison rate weights agencies by the incidents they contribute,
and Ashfell contributes about three quarters of them.

## 3. Spillover is not all or nothing

A neighbouring agency does not usually adopt the whole program. It picks up
part of it, through shared academies, transferred officers, or a supervisor
who explains it over coffee.

In [ ]:
rng = np.random.default_rng(5)
rows = []
for frac in [0.0, 0.25, 0.5, 1.0]:
    s = d.copy()
    mult = np.where((s["agency_id"] == "A012") & (s["period"] == "after"),
                    0.88 ** frac, 1.0)
    s["y"] = rng.binomial(s["n_uof"].values.astype(int), np.minimum(mult, 1.0))
    e = fit(s, keep, outcome="y")[0]
    rows.append({"share of the effect leaking to Ashfell": f"{100 * frac:.0f}%",
                 "estimate": f"{e:+.1f}%"})
print(f"  the truth throughout is {TRUTH:+.1f}%\n")
pd.DataFrame(rows).set_index("share of the effect leaking to Ashfell")

A quarter of the effect leaking costs two points. Half costs nearly five.

**There is no threshold below which spillover is safe to ignore**, and no
diagnostic in the data that distinguishes a contaminated comparison group
from a program that simply worked less well.

## 4. What can actually be done

| Response | What it buys | What it costs |
|---|---|---|
| **Ask the comparison agencies** | the only real answer | an email |
| Drop geographic neighbours | removes the likeliest route | a smaller, noisier comparison group |
| Report the estimate with and without them | bounds the exposure | nothing |
| Model the distance from a treated agency | a dose response, if distance is the route | assumes the route |
| Say nothing | | credibility, when a reviewer asks |

The second and third go together and are worth demonstrating.

In [ ]:
# The two agencies most likely to share staff and academies with the treated
# agencies are the two other large western departments.
near = ["A012", "A008"]
far = [a for a in COMPARISON if a not in near]

for label, ids in [("all seven comparison agencies", COMPARISON),
                   ("dropping the two nearest", far)]:
    s = d[d["agency_id"].isin(keep + ids)]
    e, lo, hi, _ = fit(s, keep)
    print(f"  {label:32s} {e:+6.1f}%  [{lo:+6.1f}, {hi:+6.1f}]")
print(f"  {'the truth':32s} {TRUTH:+6.1f}%")

The two estimates bracket the exposure. **Report both.** A reader who suspects
spillover can see how much it could be worth, and a reader who does not can
use the primary figure.

Dropping the two largest comparison agencies widens the interval considerably,
which is the cost, and it is the right cost to pay when a reviewer asks.

## Exercise

Spillover into the comparison group shrinks the estimate. Work out what
spillover **out of** the treated group does: agencies that were trained but
where the training did not take hold.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rng2 = np.random.default_rng(9)
    rows = []
    for frac in [1.0, 0.75, 0.5, 0.25]:
        s = d.copy()
        # one trained agency implements only a fraction of the program
        mult = np.where((s["agency_id"] == "A001") & (s["period"] == "after"),
                        0.88 ** (frac - 1.0), 1.0)
        s["y"] = rng2.poisson(s["n_uof"].values * np.maximum(mult, 0.01))
        e = fit(s, keep, outcome="y")[0]
        rows.append({"share of the program Stonewick actually implemented":
                         f"{100 * frac:.0f}%",
                     "estimate": f"{e:+.1f}%"})
    print(f"  the effect where it was implemented is always {TRUTH:+.1f}%\n")
    display(pd.DataFrame(rows).set_index(
        "share of the program Stonewick actually implemented"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Partial implementation at one treated agency pulls the estimate toward zero,
in the same direction and for the same arithmetic reason as contamination of
the comparison group.

**The two have different names and the same remedy: find out what actually
happened.** An estimate of 8 percent is consistent with a program that
reduces use of force by 8 percent everywhere, and equally consistent with one
that reduces it by 12 percent at the two thirds of agencies that ran it
properly.

That distinction matters enormously for policy and not at all for the
arithmetic, which is why implementation records belong in an evaluation
alongside the outcome data. The quantity a difference in differences
estimates is the effect of **being assigned** the program, not the effect of
receiving it, and those are the same number only when everyone assigned
received it.

</details>

---

**Next:** [Module 14: How Big an Effect Could You Have Detected?](Module_14_How_Big_An_Effect.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*